# Optofluidics · Biotech · Integral Tables · Digital Twin

Integrated photonics + microfluidics → single-cell analysis at million cells/sec.
Rare event distributions. Satellite digital twin.

In [ ]:
from sympy import *
from IPython.display import display, Markdown
import numpy as np

init_printing(use_latex='mathjax')

def step(label, expr=None):
    display(Markdown(f'**{label}**'))
    if expr is not None:
        display(expr)

def section(title):
    display(Markdown(f'---\n## {title}'))

## §1 — Integral Lookup Table: Every Form You Need

In [ ]:
section('Master integral table — derived, not memorized')

x, a, b, n_sym = symbols('x a b n', positive=True)

integrals = [
    ('Power',          x**n_sym,                   x**n_sym / (n_sym+1)),
    ('Exponential',    exp(-a*x),                  1/a),
    ('Gaussian',       exp(-a*x**2),                sqrt(pi/a)/2),   # 0 to inf
    ('x·Gaussian',     x*exp(-a*x**2),              1/(2*a)),
    ('x²·Gaussian',    x**2*exp(-a*x**2),           sqrt(pi)/(4*a**Rational(3,2))),
    ('sech²',          sech(x)**2,                  2),               # -inf to inf
    ('1/(1+x²)',       1/(1+x**2),                  pi),              # -inf to inf
    ('sin²',           sin(x)**2,                   pi),              # 0 to 2pi
    ('ln(x)',          log(x),                      symbols('x')*log(symbols('x')) - symbols('x')),
    ('x·ln(x)',        x*log(x),                    x**2/2*(log(x)-Rational(1,2))),
]

display(Markdown('### Standard integrals (verify each with SymPy below)'))

# Verify key ones
checks = [
    ('∫₀^∞ e^(-ax) dx',   integrate(exp(-a*x), (x,0,oo))),
    ('∫₀^∞ e^(-ax²) dx',  integrate(exp(-a*x**2), (x,0,oo))),
    ('∫₀^∞ x e^(-ax²) dx',integrate(x*exp(-a*x**2),(x,0,oo))),
    ('∫₀^∞ x² e^(-ax²) dx',integrate(x**2*exp(-a*x**2),(x,0,oo))),
    ('∫₋∞^∞ e^(-ax²) dx', integrate(exp(-a*x**2),(x,-oo,oo))),
    ('∫₋∞^∞ sech²(x) dx', integrate(sech(x)**2,(x,-oo,oo))),
    ('∫₋∞^∞ 1/(1+x²) dx', integrate(1/(1+x**2),(x,-oo,oo))),
]

for label, result in checks:
    step(label + ' =', simplify(result))

## §2 — Rare Event Distributions

In [ ]:
section('Rare event distributions')

display(Markdown(r'''
**When Gaussian fails**: Gaussian assumes many small independent errors.
Rare events (rogue waves, cancer cells, system failures) have **heavy tails**.

| Distribution | PDF tail | Use case |
|---|---|---|
| Gaussian | $e^{-x^2/2\sigma^2}$ | thermal noise, measurement error |
| Poisson | $\lambda^k e^{-\lambda}/k!$ | photon counting, rare events at fixed rate |
| Exponential | $\lambda e^{-\lambda x}$ | time between rare events |
| Pareto (power law) | $x^{-\alpha}$ | rogue waves, earthquakes, wealth |
| Weibull | $x^{k-1}e^{-(x/\lambda)^k}$ | device failure, fatigue |
| Log-normal | $e^{-(\ln x-\mu)^2/2\sigma^2}/x$ | biological cell sizes, stock prices |
'''))

lam, k_w, alpha_p = symbols('lambda k alpha', positive=True)

# Exponential
f_exp = lam * exp(-lam*x)
step('Exponential PDF:', f_exp)
step('Mean:', integrate(x*f_exp, (x,0,oo)))
step('P(X > t) = e^(-λt)  →  memoryless')

# Pareto — heavy tail
x_m = symbols('x_m', positive=True)  # minimum value
f_pareto = alpha_p * x_m**alpha_p / x**( alpha_p+1)
step('\nPareto PDF (x ≥ x_m):', f_pareto)
mean_pareto = integrate(x*f_pareto, (x, x_m, oo))
step('Mean:', simplify(mean_pareto))
step('Mean finite only if α > 1. Variance finite only if α > 2.')
step('Rogue wave optical intensity: α ≈ 1.5–2.5  →  variance may not exist')

# Survival function
S_pareto = integrate(f_pareto, (x, symbols('t'), oo))
step('P(X > t) =', simplify(S_pareto))
step('= (x_m/t)^α  →  power law tail, much heavier than Gaussian')

## §3 — Integrated Photonics: On-Chip Platform

In [ ]:
section('Silicon photonics on-chip')

display(Markdown(r'''
**Silicon-on-insulator (SOI)** platform:
- Si core ($n=3.45$), SiO₂ cladding ($n=1.45$), 220 nm thick
- Fabricated in CMOS foundry (same as your CPU)
- Components: waveguides, ring resonators, grating couplers, MZI, photodetectors

```
LASER (off-chip, 1550 nm)
    │ grating coupler
    ▼
  ─────────────── waveguide (450nm × 220nm)
        │
      ┌─┴─┐  ring resonator (R=5μm, Q~10⁴)
      │ ○ │  → wavelength selective filter
      └─┬─┘
        │
  ──────┴────────── MZI (Mach-Zehnder interferometer)
     arm 1   arm 2  → tunable power splitter / modulator
        │
  Ge photodetector (bandwidth > 50 GHz on-chip)
        │
  CMOS electronics (same chip in advanced nodes)
```

**Dispersive element on-chip**: spiral waveguide, 10 cm long folded into 1mm².
Anomalous dispersion engineered by waveguide geometry → on-chip TS-DFT.
'''))

n_si, n_sio2, lam_sym, w, h = symbols('n_{Si} n_{SiO2} lambda w h', positive=True)

# Effective index (approximate, for intuition)
# Full calculation requires mode solver (FEM)
n_eff_approx = n_sio2 + (n_si - n_sio2) * (1 - exp(-w*h/(lam_sym**2)))
step('n_eff (approximate) =', n_eff_approx)
step('Full solution: FEM mode solver (COMSOL, Lumerical, or open-source Meep)')

# GVD from waveguide geometry
display(Markdown(r'''
**Engineering GVD**: change waveguide width $w$ to tune $\beta_2$.
- Narrow waveguide: normal dispersion ($\beta_2 > 0$)
- Wide waveguide: anomalous ($\beta_2 < 0$) → soliton regime
- Zero-dispersion wavelength tunable from 1200–1700 nm by geometry

For TS-DFT: need $|\beta_2| \cdot L \geq 5000$ ps².  
On-chip: $\beta_2 \approx -1000$ ps²/m, $L = 5$ m (spiral) → $|\beta_2 L| = 5000$ ps² ✓
'''))

## §4 — Microfluidics + Photonics: Optofluidic Flow Cytometer

In [ ]:
section('Optofluidic flow cytometry')

display(Markdown(r'''
**Standard flow cytometry**: laser illuminates cells in flow, scatter + fluorescence measured.
Rate: ~10,000 cells/sec. Limitation: slow electronics, one cell at a time.

**Jalali optofluidic** (STEAM + microfluidics):
- Broadband pulse illuminates cell → cell scatters/absorbs wavelength-dependent
- TS-DFT maps spectrum to time → single ADC reads full spectrum per cell
- Rate: **1,000,000 cells/sec** (100× improvement)
- Can detect rare cancer cells (1 in 10⁶) in blood sample

```
Blood sample → microfluidic channel (50μm × 50μm)
                    │
                    │ ← broadband laser pulse (1550nm, 1ps)
                    │
              scattered light → optical fiber
                    │
              TS-DFT (DCF or on-chip spiral)
                    │
              photodetector + ADC
                    │
              GS phase retrieval → cell spectral fingerprint
                    │
              ML classifier → normal / cancer / pathogen
```
'''))

# Cell transit time through laser spot
v_flow, d_spot, eta_fluid, delta_P, w_ch, h_ch, L_ch = symbols(
    'v d eta DeltaP w h L', positive=True)

# Flow velocity from Hagen-Poiseuille
v_max = delta_P * h_ch**2 / (8*eta_fluid*L_ch)  # max velocity (centerline)
step('Max flow velocity (H-P): v_max = ΔP·h²/(8ηL) =', v_max)

# Transit time
t_transit = d_spot / v_flow
step('Cell transit time through laser spot: t = d_spot/v =', t_transit)

# Numeric
print('\nTypical values:')
print('  flow velocity v = 1 m/s')
print('  laser spot d = 50 μm')
print(f'  transit time = {50e-6/1*1e6:.0f} μs')
print(f'  laser rep rate = 100 MHz → {100e6 * 50e-6:.0f} pulses per transit')
print(f'  → {100e6*50e-6:.0f} spectra per cell → 3D image reconstruction')

# Rare cell detection
display(Markdown(r'''
**Rare event statistics**: circulating tumor cells (CTCs) at 1 per 10⁶ blood cells.

At 10,000 cells/sec (standard): need 100 sec per CTC on average.
At 1,000,000 cells/sec (STEAM): need 1 sec per CTC → **clinically useful**.

Poisson statistics: probability of seeing $k$ CTCs in $n$ cells, rate $\lambda = n/10^6$:
$$P(k) = \frac{\lambda^k e^{-\lambda}}{k!}$$
'''))

lam_ctc = symbols('lambda', positive=True)
k_ctc = symbols('k', integer=True, nonnegative=True)
P_ctc = lam_ctc**k_ctc * exp(-lam_ctc) / factorial(k_ctc)
step('P(k CTCs) =', P_ctc)

# P(at least one) in 1M cells
P_zero = P_ctc.subs([(lam_ctc, 1), (k_ctc, 0)])
P_atleast_one = 1 - P_zero
print(f'\nIn 1M cells scanned (λ=1): P(at least one CTC) = {float(P_atleast_one.evalf()):.4f}')

## §5 — Voltage and Sensing: Apple Watch to Satellite

In [ ]:
section('Sensing hierarchy: wrist to orbit')

display(Markdown(r'''
| Platform | Sensor | Signal | Bandwidth | Power |
|---|---|---|---|---|
| Apple Watch | PPG (optical HR) | $\Delta R/R \sim 10^{-3}$ | 1–10 Hz | 1 mW |
| Apple Watch | ECG (electrical) | 1 mV, 12-lead equiv | 0.05–150 Hz | 0.1 mW |
| Medical flow cytometer | optical scatter | nW per cell | 10 kHz | 50 mW |
| STEAM cytometer | broadband optical | pJ per pulse | 100 MHz | 1 W |
| OTDR ship | pulsed optical | nJ backscatter | 1 GHz | 100 mW |
| Satellite SAR | microwave radar | kW transmitted | 100 MHz | 1 kW |
| Hypersonic sensor (proposed) | optical through plasma | fJ scattered | 10 GHz | 10 mW |

**Voltage measurement** is always: $V = IR$ or $V = \int E\cdot dl$.
The physics is the same at every scale — only noise floor and bandwidth change.
'''))

# PPG signal model
t_sym, HR, A_dc, A_ac = symbols('t f_HR A_dc A_ac', positive=True)
I_ppg = A_dc + A_ac * sin(2*pi*HR*t_sym)  # simplified
step('PPG intensity model: I(t) =', I_ppg)
step('Perfusion index PI = A_ac/A_dc  (typical: 0.02–2%)')

# Shot noise on PPG
q_e, I_photo, B_BW = symbols('q I_photo B', positive=True)
i_shot = sqrt(2*q_e*I_photo*B_BW)
step('Shot noise on photocurrent: i_shot = √(2qIB) =', i_shot)
step('SNR_shot = I_ac/i_shot = A_ac/(√(2q·A_dc·B))')
print('\nApple Watch PPG: I_dc~1μA, B~10Hz, A_ac~10nA')
print(f'  i_shot = {np.sqrt(2*1.6e-19*1e-6*10)*1e12:.3f} pA')
print(f'  SNR = {10e-9/np.sqrt(2*1.6e-19*1e-6*10):.0f}  → plenty of margin')

## §6 — Satellite Digital Twin and Controls

In [ ]:
section('Digital twin: satellite orbital mechanics')

display(Markdown(r'''
A **digital twin** is a real-time simulation synchronized to a physical system.
Sensor data updates the model; the model predicts future state and optimal control.

```
PHYSICAL SATELLITE
  IMU, GPS, star tracker → state: [r, v, q, ω]
         │ telemetry (1 Hz–100 Hz)
         ▼
  DIGITAL TWIN (ground or onboard)
  ┌─────────────────────────────────┐
  │  Orbital propagator (RK4/SGP4) │
  │  Attitude dynamics (Euler eqs) │
  │  Sensor models + noise         │
  │  Kalman filter (state est.)    │
  └──────────────┬──────────────────┘
                 │ predicted state
                 ▼
  CONTROLLER (LQR / MPC)
  → thruster commands, reaction wheel torques
                 │
                 ▼
  PHYSICAL SATELLITE (closes the loop)
```
'''))

# Orbital mechanics: vis-viva
G, M_earth, r_orbit, a_sma = symbols('G M r a', positive=True)
v_orbit = sqrt(G*M_earth*(2/r_orbit - 1/a_sma))
step('Vis-viva equation: v = √(GM(2/r − 1/a)) =', v_orbit)

# Circular orbit
v_circ = sqrt(G*M_earth/r_orbit)
step('Circular orbit: v_c = √(GM/r) =', v_circ)

# Numeric: LEO at 400km
G_val   = 6.674e-11
M_val   = 5.972e24
R_earth = 6.371e6
h_leo   = 400e3
r_val   = R_earth + h_leo
v_val   = np.sqrt(G_val*M_val/r_val)
T_val   = 2*np.pi*r_val/v_val
print(f'LEO (400 km):')
print(f'  v = {v_val/1000:.3f} km/s')
print(f'  T = {T_val/60:.1f} min')
print(f'  Mach = {v_val/343:.1f}')

# Euler's equations for attitude
display(Markdown(r'''
**Euler's rigid body equations** (attitude dynamics):
$$I_1\dot{\omega}_1 + (I_3-I_2)\omega_2\omega_3 = \tau_1$$
$$I_2\dot{\omega}_2 + (I_1-I_3)\omega_3\omega_1 = \tau_2$$
$$I_3\dot{\omega}_3 + (I_2-I_1)\omega_1\omega_2 = \tau_3$$

$I_i$ = principal moments of inertia, $\omega_i$ = body angular rates, $\tau_i$ = torques.

**Digital twin update**: integrate these ODEs at 1 kHz on ground,
compare to IMU data, Kalman filter corrects model drift.
Model predicts attitude 10 sec ahead → controller pre-commands thrusters.
'''))

# Kalman filter one-step
display(Markdown(r'''
**Kalman filter** (linear, discrete):

Predict: $\hat{x}^- = F\hat{x},\quad P^- = FPF^T + Q$

Update: $K = P^-H^T(HP^-H^T+R)^{-1}$
$\quad\quad\quad \hat{x} = \hat{x}^- + K(z - H\hat{x}^-)$
$\quad\quad\quad P = (I-KH)P^-$

$Q$ = process noise, $R$ = measurement noise, $K$ = Kalman gain.
Optimal when noise is Gaussian — for non-Gaussian (rogue events),
use particle filter or robust Kalman.
'''))

## §7 — Centrifuge: Angular Momentum and Separation

In [ ]:
section('Centrifuge physics')

omega_c, r_c, rho_p, rho_f, eta_c, d_p = symbols('omega r rho_p rho_f eta d', positive=True)

# Centrifugal acceleration
a_cf = omega_c**2 * r_c
step('Centrifugal acceleration a = ω²r =', a_cf)
step('G-force = a/g = ω²r/9.81')

# Stokes sedimentation velocity in centrifuge
g_sym = symbols('g', positive=True)
v_sed = d_p**2 * (rho_p - rho_f) * a_cf / (18*eta_c)
step('Stokes sedimentation velocity: v = d²(ρ_p−ρ_f)ω²r/(18η) =', v_sed)

# Numeric: blood cell separation
rpm_val = 3000  # rpm
omega_val = rpm_val * 2*np.pi/60
r_val_c = 0.1  # 10 cm
g_force = omega_val**2 * r_val_c / 9.81
print(f'Blood centrifuge: {rpm_val} RPM, r=10 cm')
print(f'  ω = {omega_val:.1f} rad/s')
print(f'  G-force = {g_force:.0f}×g')

# RBC sedimentation
d_rbc = 8e-6   # 8 μm
rho_rbc = 1100  # kg/m³
rho_plasma = 1025
eta_blood = 1.2e-3  # Pa·s
v_rbc = d_rbc**2*(rho_rbc-rho_plasma)*omega_val**2*r_val_c/(18*eta_blood)
print(f'  RBC sedimentation velocity = {v_rbc*1e3:.3f} mm/s')
print(f'  Time to pellet (10mm) = {10e-3/v_rbc/60:.1f} min')

display(Markdown(r'''
**Lab-on-chip centrifuge** (microfluidic CD): spin a plastic disc,
centrifugal force drives fluid through channels without pumps.
Combine with on-chip photonics → integrated blood analysis.
The optical readout is your TDGSA sensor, miniaturized.
'''))